**D. PRO: Финансовый щит (Сбер)**  


**Описание задачи**

Банк хочет в режиме, близком к реальному времени, определять, какие транзакции с высокой вероятностью являются мошенническими, чтобы вовремя блокировать операции или запрашивать дополнительную верификацию.

**Задание**

Ваша задача — по данным о транзакции определить, является ли она мошеннической. Обратите внимание: данные поступают потоком, а поведение клиентов и злоумышленников меняется со временем (дрейф).

Вам необходимо:

- реализовать и обучить модель классификации, которая на основе имеющихся данных сможет сделать предсказание о том, является ли транзакция мошеннической;
- получить предсказания на тестовых данных;
- загрузить `.csv` файл с предсказаниями.



**Данные**

Набор данных вы можете скачать, перейдя по ссылке.

Общий набор был поделен на обучающую, открытую тестовую и закрытую тестовую выборки в отношении 5:1:1. Обучающий набор содержит 10000 строк, каждая из тестовых выборок - по 2000 строк.

В наборе данных вы найдете файлы:

- `train.csv` - обучающая выборка;
- `test_public.csv` - открытая тестовая выборка;
- `sample_submission.csv` - пример предсказаний.

**Признаки объектов:**

| Название признака | Описание признака |
|-------------------|-------------------|
| customer_id       | Уникальный идентификатор клиента               |
| age               | Возраст клиента                               |
| tenure_months     | Срок обслуживания клиента в банке (в месяцах)   |
| risk_profile      | Риск-профиль клиента (низкий/средний/высокий)  |
| home_country      | Страна проживания клиента                      |
| tx_id             | Уникальный идентификатор транзакции            |
| tx_datetime       | Дата и время транзакции                        |
| channel           | Канал проведения транзакции (онлайн, POS, ATM и т.д.) |
| mcc               | Код категории торговца (Merchant Category Code) |
| country           | Страна, в которой была проведена транзакция    |
| amount            | Сумма транзакции                               |
| device_trust_score| Оценка доверия к устройству                    |
| distance_km       | Расстояние между локацией транзакции и домашней/обычной локацией клиента |
| attempts_1h       | Количество попыток транзакций с карты клиента за последний час |
| is_night          | Флаг, указывающий, что транзакция проведена ночью |
| is_weekend        | Флаг, указывающий, что транзакция проведена в выходные дни |
| is_international  | Флаг, указывающий, что транзакция международная |
| avg_amount_7d     | Средняя сумма транзакций клиента за последние 7 дней |
| std_amount_7d     | Стандартное отклонение суммы транзакций клиента за последние 7 дней |
| ratio_to_avg      | Отношение суммы текущей транзакции к средней сумме за последние 7 дней |
| prev_fraud_30d    | Количество мошеннических транзакций у клиента за последние 30 дней |
| fraud             | Целевая переменная: флаг мошеннической транзакции (1 — мошенничество, 0 — нет) |

---

**Метрика**

PR-AUC.

---

**Формат вывода**

Файл `.csv` с 2 столбцами: `tx_id` и `fraud`. В первой строке указаны соответствующие заголовки, в остальных - идентификатор транзакции и значение `fraud` 0/1, где 1 - мошенническая транзакция, а 0 - нет. Смотрите образец в файле `sample_submission.csv`.

---

In [34]:
import pandas as pd
import numpy as np

In [35]:
train_df = pd.read_csv("train.csv")

In [36]:
print(train_df.head().to_markdown())

|    | customer_id   |   age |   tenure_months | risk_profile   | home_country   | tx_id                                | tx_datetime      | channel   | mcc             | country   |   amount |   device_trust_score |   distance_km |   attempts_1h |   is_night |   is_weekend |   is_international |   avg_amount_7d |   std_amount_7d |   ratio_to_avg |   prev_fraud_30d |   fraud |
|---:|:--------------|------:|----------------:|:---------------|:---------------|:-------------------------------------|:-----------------|:----------|:----------------|:----------|---------:|---------------------:|--------------:|--------------:|-----------:|-------------:|-------------------:|----------------:|----------------:|---------------:|-----------------:|--------:|
|  0 | C00851        |    47 |             113 | low            | RU             | 3e74b0cd-1491-40f4-a3c6-47378ff63a2a | 2025-07-01T11:29 | atm       | food            | RU        |  4202.12 |                0.92  |          7.41 |        

In [37]:
test_df = pd.read_csv("test_public.csv")

In [38]:
print(test_df.head().to_markdown())

|    | customer_id   |   age |   tenure_months | risk_profile   | home_country   | tx_id                                | tx_datetime      | channel   | mcc             | country   |   amount |   device_trust_score |   distance_km |   attempts_1h |   is_night |   is_weekend |   is_international |   avg_amount_7d |   std_amount_7d |   ratio_to_avg |   prev_fraud_30d |
|---:|:--------------|------:|----------------:|:---------------|:---------------|:-------------------------------------|:-----------------|:----------|:----------------|:----------|---------:|---------------------:|--------------:|--------------:|-----------:|-------------:|-------------------:|----------------:|----------------:|---------------:|-----------------:|
|  0 | C00465        |    76 |              36 | low            | RU             | f4c5a182-c6c0-436e-ab7d-be5cebcb6f82 | 2025-08-04T07:25 | ecom      | food            | DE        |  3223.73 |                0.883 |          9.62 |             0 |          0 

In [39]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         2000 non-null   object 
 1   age                 2000 non-null   int64  
 2   tenure_months       2000 non-null   int64  
 3   risk_profile        2000 non-null   object 
 4   home_country        2000 non-null   object 
 5   tx_id               2000 non-null   object 
 6   tx_datetime         2000 non-null   object 
 7   channel             2000 non-null   object 
 8   mcc                 2000 non-null   object 
 9   country             2000 non-null   object 
 10  amount              2000 non-null   float64
 11  device_trust_score  2000 non-null   float64
 12  distance_km         2000 non-null   float64
 13  attempts_1h         2000 non-null   int64  
 14  is_night            2000 non-null   int64  
 15  is_weekend          2000 non-null   int64  
 16  is_int

In [40]:
# Получаем уникальные customer_id из каждой выборки
train_ids = set(train_df['customer_id'])
test_ids = set(test_df['customer_id'])

# Находим пересечение
common_ids = train_ids & test_ids

# Выводим результаты
print(f"Количество уникальных клиентов в train: {len(train_ids)}")
print(f"Количество уникальных клиентов в test: {len(test_ids)}")
print(f"Количество общих клиентов: {len(common_ids)}")


Количество уникальных клиентов в train: 2456
Количество уникальных клиентов в test: 1381
Количество общих клиентов: 1357


In [41]:
train_ids = set(train_df['customer_id'])
test_ids = set(test_df['customer_id'])

# Разница: ID из test, отсутствующие в train
missing_in_train = test_ids - train_ids

print(f"Количество пользователей из test, отсутствующих в train: {len(missing_in_train)}")


Количество пользователей из test, отсутствующих в train: 24


In [42]:
import pandas as pd

# Разделяем столбцы по типам
cat_cols = train_df.select_dtypes(include=['object']).columns
num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns

print("=== УНИКАЛЬНЫЕ ЗНАЧЕНИЯ (столбцы типа object) ===\n")
for col in cat_cols:
    unique_vals = train_df[col].unique()
    print(f"{col} ({len(unique_vals)} уникальных):")
    # Если слишком много уникальных — показываем первые 10 + сообщение
    if len(unique_vals) > 10:
        print(list(unique_vals[:10]), "... и ещё", len(unique_vals) - 10, "значений")
    else:
        print(list(unique_vals))
    print()




=== УНИКАЛЬНЫЕ ЗНАЧЕНИЯ (столбцы типа object) ===

customer_id (2456 уникальных):
['C00851', 'C01754', 'C00631', 'C02354', 'C02027', 'C01324', 'C02298', 'C01410', 'C02423', 'C00442'] ... и ещё 2446 значений

risk_profile (3 уникальных):
['low', 'mid', 'high']

home_country (6 уникальных):
['RU', 'DE', 'CN', 'US', 'KZ', 'TR']

tx_id (10000 уникальных):
['3e74b0cd-1491-40f4-a3c6-47378ff63a2a', 'cb40cb78-ede3-4794-9e84-b1a5ac677eb5', '264a33d9-51c1-4a52-a25a-7ac65fe033a3', '4d342736-4324-4d8b-9396-bf6a35f0d3cc', 'd2d0c230-9f71-4912-b37c-302919c538dc', 'a357e714-dae5-46e9-a778-66b374a6c110', '73c6169b-4ca6-4c0c-b2fd-2e4bd7dfd98c', 'b83f2147-169a-4105-aff8-68bf07977bd1', '1fca454e-b170-46aa-9b84-42db29198a15', '5a2f2197-094c-409c-acf6-30c6979d128f'] ... и ещё 9990 значений

tx_datetime (9276 уникальных):
['2025-07-01T11:29', '2025-07-09T19:23', '2025-06-15T10:53', '2025-06-30T13:39', '2025-06-13T16:50', '2025-07-02T00:00', '2025-06-02T19:21', '2025-07-07T22:44', '2025-06-16T10:21', '2025-06

In [43]:
print("\n=== ОПИСАТЕЛЬНАЯ СТАТИСТИКА (числовые столбцы) ===\n")
# Выводим базовую статистику для числовых столбцов
print(train_df[num_cols].describe().round(2).T)


=== ОПИСАТЕЛЬНАЯ СТАТИСТИКА (числовые столбцы) ===

                      count     mean      std     min      25%      50%  \
age                 10000.0    48.22    18.04   18.00    32.00    48.00   
tenure_months       10000.0    59.81    34.11    1.00    31.00    60.00   
amount              10000.0  5019.19  5510.17  227.34  1564.09  2953.48   
device_trust_score  10000.0     0.77     0.27    0.00     0.79     0.88   
distance_km         10000.0     7.94     7.86    0.00     2.30     5.53   
attempts_1h         10000.0     0.28     0.55    0.00     0.00     0.00   
is_night            10000.0     0.29     0.45    0.00     0.00     0.00   
is_weekend          10000.0     0.29     0.45    0.00     0.00     0.00   
is_international    10000.0     0.52     0.50    0.00     0.00     1.00   
avg_amount_7d       10000.0  2728.55  1332.08  100.00  1768.09  2337.85   
std_amount_7d       10000.0   707.82   546.02    1.00   328.23   518.60   
ratio_to_avg        10000.0     2.39     3.97  

In [44]:
# Пример списка числовых колонок (подставьте свои)
num_cols = [
    'age', 'tenure_months', 'amount', 'device_trust_score',
    'distance_km', 'attempts_1h', 'is_night', 'is_weekend',
    'is_international',
    'ratio_to_avg', 'prev_fraud_30d'
]

In [45]:
train_df.columns

Index(['customer_id', 'age', 'tenure_months', 'risk_profile', 'home_country',
       'tx_id', 'tx_datetime', 'channel', 'mcc', 'country', 'amount',
       'device_trust_score', 'distance_km', 'attempts_1h', 'is_night',
       'is_weekend', 'is_international', 'avg_amount_7d', 'std_amount_7d',
       'ratio_to_avg', 'prev_fraud_30d', 'fraud'],
      dtype='object')

In [46]:
# Создаем новую колонку с объединёнными текстовыми признаками для train_df
train_df['combined_text'] = (
    train_df['risk_profile'].astype(str) +
    ' ' + train_df['home_country'].astype(str) +
    ' ' + train_df['channel'].astype(str) +
    ' ' + train_df['mcc'].astype(str) +
    ' ' + train_df['country'].astype(str) +
    ' ' + train_df['customer_id'].astype(str)
)    

In [47]:
# Для test_df
test_df['combined_text'] = (
    test_df['risk_profile'].astype(str) + ' ' +
    test_df['home_country'].astype(str) + ' ' +
    test_df['channel'].astype(str) + ' ' +
    test_df['mcc'].astype(str) + ' ' +
    test_df['country'].astype(str) + ' ' +
    test_df['customer_id'].astype(str)

)


In [48]:
print("train_df['combined_text'] (первые 5 строк):")
print(train_df['combined_text'].head())

print("\ntest_df['combined_text'] (первые 5 строк):")
print(test_df['combined_text'].head())


train_df['combined_text'] (первые 5 строк):
0               low RU atm food RU C00851
1    mid RU pos cash_withdrawal US C01754
2            low RU pos apparel RU C00631
3               mid RU pos food TR C02354
4      low DE pos digital_goods RU C02027
Name: combined_text, dtype: object

test_df['combined_text'] (первые 5 строк):
0              low RU ecom food DE C00465
1      low RU pos digital_goods RU C01753
2             mid RU pos travel TR C01637
3    low RU pos cash_withdrawal KZ C00245
4        mid RU pos electronics RU C02239
Name: combined_text, dtype: object


In [49]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [51]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder


# Определяем препроцессор для текстовых данных
text_transformer = Pipeline([
    ('tfidf', TfidfVectorizer())
])


# Препроцессор для числовых данных
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])


# Объединяем оба трансформера
preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_transformer, 'combined_text'),
        ('numeric', numeric_transformer, num_cols)
    ])

# Полная модель
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [52]:
# Определяем признаки (X) и целевую переменную (y)
X = train_df[[
    'customer_id',
    'combined_text',           # наша объединённая текстовая колонка
    'age',                    # числовой признак
    'tenure_months',         # числовой признак
    'amount',                 # числовой признак
    'device_trust_score',    # числовой признак
    'distance_km',           # числовой признак
    'attempts_1h',           # числовой признак
    'is_night',              # бинарный признак
    'is_weekend',            # бинарный признак
    'is_international',       # бинарный признак
    'ratio_to_avg',          # числовой признак
    'prev_fraud_30d'         # бинарный признак
]]
y = train_df['fraud']  # целевая переменная

# Разделяем данные на обучающую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y,
    test_size=0.2,           # 20% данных для валидации
    random_state=42,      # воспроизводимость разбиения
    stratify=y            # сохранение баланса классов в выборках
)


In [53]:
# Создаем полную модель
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])

# Обучение модели
model.fit(X_train, y_train)

# Прогнозирование и расчет F1-метрики
val_preds = model.predict(X_val)
print(f'F1-score на валидации: {f1_score(y_val, val_preds, average="macro")}')

F1-score на валидации: 0.4836044410018074


In [54]:
# Простая модель для теста
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Обучение модели
model.fit(X_train, y_train)

# Прогнозирование и расчёт F1-метрики
val_preds = model.predict(X_val)
print(f'F1-score на валидации: {f1_score(y_val, val_preds, average="macro")}')

F1-score на валидации: 0.5769391489790919


In [55]:
from sklearn.metrics import precision_recall_curve, average_precision_score, classification_report, f1_score

# Предсказание вероятностей для PR-AUC
y_val_proba = model.predict_proba(X_val)[:, 1]

# 1. PR-AUC
pr_auc = average_precision_score(y_val, y_val_proba)
print(f'PR-AUC на валидации: {pr_auc:.4f}')

# 2. F1-score (макро)
val_preds = model.predict(X_val)
f1_macro = f1_score(y_val, val_preds, average='macro')
print(f'F1-score (макро) на валидации: {f1_macro:.4f}')

# 3. Classification Report
print("\nClassification Report:")
print(classification_report(y_val, val_preds, target_names=['Не мошенничество', 'Мошенничество']))

PR-AUC на валидации: 0.2250
F1-score (макро) на валидации: 0.5769

Classification Report:
                  precision    recall  f1-score   support

Не мошенничество       0.96      0.84      0.90      1873
   Мошенничество       0.17      0.50      0.26       127

        accuracy                           0.82      2000
       macro avg       0.57      0.67      0.58      2000
    weighted avg       0.91      0.82      0.85      2000



In [57]:
model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [58]:
x = test_df[[
    'customer_id',
    'combined_text',           # наша объединённая текстовая колонка
    'age',                    # числовой признак
    'tenure_months',         # числовой признак
    'amount',                 # числовой признак
    'device_trust_score',    # числовой признак
    'distance_km',           # числовой признак
    'attempts_1h',           # числовой признак
    'is_night',              # бинарный признак
    'is_weekend',            # бинарный признак
    'is_international',       # бинарный признак
    'ratio_to_avg',          # числовой признак
    'prev_fraud_30d'         # бинарный признак
]]

In [59]:
# Прогнозирование и оценка модели
y_pred = model.predict(x)

In [60]:
submission_df = pd.DataFrame({
    'tx_id': test_df['tx_id'],
    'fraud': y_pred
})

In [61]:
# Убедимся, что целевой признак имеет нужный тип данных (целочисленный):
submission_df['fraud'] = submission_df['fraud'].astype(int)

# Просмотрим первые строки получившегося DataFrame:
print(submission_df.head())

                                  tx_id  fraud
0  f4c5a182-c6c0-436e-ab7d-be5cebcb6f82      0
1  9d20d6f2-eead-45a9-a07b-8996286f0174      0
2  aa74d5d7-34e1-410e-9a76-d5d8894cf603      1
3  1b15098b-717b-44cb-8073-432c2f3e06ac      0
4  126678f1-9b95-40b0-8d0f-6e823a0d90e6      0


In [62]:
submission_df.to_csv('4_n_submission.csv', index=False)